# 프로게스테론 De Novo 단백질 설계
### LigandMPNN 파이프라인 (Step 3) — v2

포트폴리오: 이규리 교수님 랩 (KAIST AIPD Lab) 지원용  
작성자: 김장현 | GitHub: jhkwin00  
기반: LigandMPNN 공식 레포 (dauparas/LigandMPNN)  
논문: Dauparas, Lee (이규리) et al. Nature Methods 2025

---

### v1 → v2 수정 사항

| # | 문제 (v1) | 원인 | 수정 (v2) |
|---|-----------|------|-----------|
| 1 | num_ligand_res=0 | --chains_to_design 미지정으로 STR(Chain B) 미인식 | --chains_to_design "A" 추가 |
| 2 | 결과 파싱 실패 (서열 0개) | 헤더가 score= 아닌 overall_confidence= 형식 | overall_confidence 기준으로 파싱 수정 |
| 3 | Drive 저장 경로 충돌 가능 | 재실행 시 ligandmpnn/ 덮어쓰기 | ligandmpnn_v2/ 로 분리 |
| 4 | ColabFold 입력 준비 없음 | 셀 미포함 | seq_rec 필터 + rank별 .fasta 생성 셀 추가 |

---

이 단계에서 하는 일:
```
입력: sample_0.pdb (RFdiffusionAA로 생성한 de novo backbone + 프로게스테론 좌표)
        |
        v
LigandMPNN (--chains_to_design A 로 Chain B = STR 리간드 명시)
  backbone 3D 구조 + 프로게스테론(STR, Chain B) 원자 좌표를 동시에 보고
  프로게스테론 결합에 최적화된 아미노산 서열 설계
        |
        v
출력: FASTA 파일 (서열 30개, num_ligand_res=46 확인)
  -> ColabFold 입력용 rank별 .fasta 생성
  -> 다음 단계(ColabFold)에서 pAE 필터링
```

전체 파이프라인:

| 단계 | 상태 | 도구 | 내용 |
|------|------|------|------|
| Step 1 | 완료 | PyMOL | PDB 1A28 에서 STR 좌표 추출 |
| Step 2 | 완료 | RFdiffusionAA | 프로게스테론 주변 backbone 생성 |
| Step 3 | 이번 세션 | LigandMPNN | backbone -> 서열 설계 (30개) |
| Step 4 | 예정 | ColabFold | 서열 -> pAE 필터 -> 상위 3~5개 |
| Step 5 | 예정 | PyMOL | 포켓 시각화 + 수소결합 분석 |
| Step 6 | 예정 | GitHub | README + 포트폴리오 정리 |

---
## 1단계: LigandMPNN 설치

공식 레포: https://github.com/dauparas/LigandMPNN  
이규리 교수님이 핵심 개발자로 참여한 도구 (논문 2저자: Gyu Rie Lee).  

RFdiffusionAA와 달리 Singularity 컨테이너 없이 pip install만으로 환경 구성 가능.  

**np.int 호환성 문제:**  
LigandMPNN 내부 openfold 코드가 numpy 구버전 문법(np.int)을 사용함.  
Python 3.12 + 최신 numpy 환경에서는 np.int가 삭제되어 에러 발생.  
→ sed로 해당 파일의 np.int를 int로 직접 수정해서 해결.

In [ ]:
# LigandMPNN 공식 레포 클론
!git clone https://github.com/dauparas/LigandMPNN.git
%cd LigandMPNN

# 의존 라이브러리 설치
# prody: PDB 파일 파싱에 필요
# ml_collections: openfold 의존성
!pip install prody ml_collections --quiet

# np.int 호환성 수정
# openfold/np/residue_constants.py 에서 np.int -> int 로 치환
# sed -i: 파일 내용을 직접 찾아서 바꾸는 명령어 (-i: in-place 수정)
!sed -i 's/dtype=np\.int\b/dtype=int/g' /content/LigandMPNN/openfold/np/residue_constants.py

# 모델 가중치 다운로드
# get_model_params.sh: ligand_mpnn 등 모델 가중치를 model_params/ 폴더에 저장
!bash get_model_params.sh "./model_params"

# 설치 확인
!python run.py --help 2>&1 | head -5
print('\nLigandMPNN 설치 완료')

---
## 2단계: Drive 마운트 + sample pdb 복사 + PDB 검증

RFdiffusionAA로 생성한 sample_*.pdb 파일을 Drive에서 가져온다.  

**LigandMPNN 입력 PDB 조건:**  
- ATOM 레코드 (Chain A): de novo backbone (단백질 주쇄)  
- HETATM STR 레코드 (Chain B): 프로게스테론 원자 좌표 (46개)  
- RFdiffusionAA 출력 파일이 이 조건을 자동으로 만족함  

**[v2 추가] PDB 검증:**  
복사 후 HETATM STR 레코드가 실제로 있는지 확인한다.  
num_ligand_res=0 방지를 위해 Chain 구조를 미리 점검.

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

DRIVE_RF_OUTPUT = '/content/drive/MyDrive/RFdiffusionAA/output/progesterone'
# [v2 수정] ligandmpnn_v2/ 로 저장 경로 분리 (v1 결과와 충돌 방지)
DRIVE_LM_OUTPUT = '/content/drive/MyDrive/RFdiffusionAA/output/ligandmpnn_v2'

os.makedirs('inputs', exist_ok=True)
os.makedirs(DRIVE_LM_OUTPUT, exist_ok=True)

# Drive에서 완료된 sample pdb 복사
print('=== Drive에서 sample pdb 복사 ===')
copied = 0
for i in range(10):
    src = f'{DRIVE_RF_OUTPUT}/sample_{i}.pdb'
    dst = f'inputs/sample_{i}.pdb'
    if os.path.exists(src):
        shutil.copy(src, dst)
        size_kb = os.path.getsize(dst) / 1e3
        print(f'  sample_{i}.pdb ({size_kb:.1f} KB) 복사 완료')
        copied += 1

if copied == 0:
    print('  복사할 파일 없음 -> RFdiffusionAA 결과 확인 필요')
else:
    print(f'\n총 {copied}개 복사 완료')

# [v2 추가] PDB 검증: ATOM / HETATM STR 레코드 확인
# num_ligand_res=0 문제 사전 방지
print('\n=== PDB 검증 (ATOM / HETATM 확인) ===')
for fname in sorted(os.listdir('inputs')):
    if not fname.endswith('.pdb'):
        continue
    with open(f'inputs/{fname}') as f:
        lines = f.readlines()
    atom_lines = [l for l in lines if l.startswith('ATOM')]
    hetatm_lines = [l for l in lines if l.startswith('HETATM') and 'STR' in l]
    status = '✅' if hetatm_lines else '❌ STR 없음!'
    print(f'  {fname}: ATOM {len(atom_lines)}개, HETATM STR {len(hetatm_lines)}개 {status}')

---
## 3단계: LigandMPNN 실행

각 backbone당 30개 서열 생성.  

**파라미터 설명 (run.py --help 기준):**

| 파라미터 | 값 | 의미 |
|----------|-----|------|
| --model_type | ligand_mpnn | 소분자 원자 좌표를 인식하는 모델 |
| --temperature | 0.1 | 서열 샘플링 온도. 낮을수록 안정적, 높을수록 다양. |
| --batch_size | 1 | 한 번에 처리할 서열 수. Colab 메모리에 맞게 1 |
| --number_of_batches | 30 | batch 반복 횟수. batch_size=1이면 서열 30개 생성 |
| --seed | 111 | 재현성 확보 |
| --save_stats | 1 | 각 서열의 점수 통계 저장 |
| **--chains_to_design** | **"A"** | **[v2 핵심 수정] Chain A(단백질)만 서열 설계** |

**[v2 핵심 수정] --chains_to_design "A" 추가 이유:**  
sample_0.pdb 구조:
- Chain A: de novo 단백질 backbone (ATOM 레코드, 655개 원자)
- Chain B: 프로게스테론 STR (HETATM 레코드, 46개 원자)

이 파라미터 없이 실행하면 LigandMPNN이 Chain B까지 단백질로 파싱하려다 STR을 리간드로 인식하지 못함.  
→ num_ligand_res=0, overall_confidence 0.30~0.38 (비정상적으로 낮음)  

"A" 명시 시 Chain B(STR)는 자동으로 소분자 리간드로 처리됨.  
→ num_ligand_res=46, overall_confidence 0.40~0.55 (정상 범위) 기대.

In [ ]:
import os, time, shutil

sample_files = sorted([f for f in os.listdir('inputs') if f.endswith('.pdb')])

if not sample_files:
    print('inputs/ 폴더에 PDB 파일 없음 -> 2단계 다시 실행')
else:
    print(f'처리할 파일: {sample_files}\n')

    for fname in sample_files:
        name = fname.replace('.pdb', '')   # 'sample_0'
        pdb_path = f'inputs/{fname}'
        out_folder = f'outputs/{name}'
        drive_out = f'{DRIVE_LM_OUTPUT}/{name}'

        os.makedirs(out_folder, exist_ok=True)
        os.makedirs(drive_out, exist_ok=True)

        print(f'--- {name} 서열 설계 중 ---')
        start = time.time()

        # [v2 핵심 수정] --chains_to_design "A" 추가
        # Chain A(단백질)만 서열 설계 대상으로 지정
        # Chain B(STR 프로게스테론)는 자동으로 리간드로 인식 -> num_ligand_res=46 기대
        !python run.py \
            --model_type "ligand_mpnn" \
            --seed 111 \
            --pdb_path {pdb_path} \
            --out_folder {out_folder} \
            --temperature 0.1 \
            --batch_size 1 \
            --number_of_batches 30 \
            --save_stats 1 \
            --chains_to_design "A"

        elapsed = time.time() - start
        print(f'  완료: {elapsed:.1f}초')

        # 완료 즉시 Drive 백업 (런타임 끊겨도 보존)
        if os.path.exists(f'{out_folder}/seqs'):
            shutil.copytree(f'{out_folder}/seqs', f'{drive_out}/seqs', dirs_exist_ok=True)
            print(f'  Drive 백업 완료: {drive_out}/seqs')

    print('\n전체 완료')

---
## 4단계: 결과 확인

**[v2 수정] 파싱 기준 변경:**  
v1은 score= 필드를 파싱했으나 실제 헤더 형식은 overall_confidence= 임.  
→ overall_confidence 기준으로 파싱 수정 (높을수록 좋음).  

**[v2 추가] num_ligand_res 검증:**  
첫 번째 헤더에서 num_ligand_res 확인.  
46이면 STR 원자 46개 정상 인식. 0이면 --chains_to_design 문제.  

**지표 설명:**  
- overall_confidence: 높을수록 좋음. 정상 범위 0.40~0.55  
- ligand_confidence: 리간드 결합 포켓 신뢰도. 1.0에 가까울수록 좋음  
- seq_rec: backbone 서열 회복률. 0.3~0.6이 일반적

In [ ]:
import os, glob

print('=== LigandMPNN 결과 요약 ===')

all_results = []
fasta_files = sorted(glob.glob('outputs/*/seqs/*.fa'))

if not fasta_files:
    print('FASTA 파일 없음 -> 3단계 실행 완료 여부 확인')
else:
    for fa_path in fasta_files:
        sample_name = fa_path.split('/')[1]  # 'sample_0'
        print(f'\n[ {sample_name} ]')

        with open(fa_path) as f:
            lines = f.readlines()

        # [v2 추가] 첫 번째 헤더에서 num_ligand_res 확인
        # num_ligand_res=46 이면 STR 정상 인식, 0 이면 --chains_to_design 문제
        first_header = lines[0].strip()
        try:
            num_lig = int([x for x in first_header.split(',') if 'num_ligand_res=' in x][0].split('=')[1])
            status = '✅ 정상' if num_lig > 0 else '❌ 리간드 미인식 (--chains_to_design 확인)'
            print(f'  num_ligand_res: {num_lig} {status}')
        except:
            print('  num_ligand_res 파싱 실패')

        seqs = []
        for i, line in enumerate(lines):
            if line.startswith('>') and 'id=' in line:
                header = line.strip()
                seq = lines[i+1].strip() if i+1 < len(lines) else ''
                try:
                    # [v2 수정] overall_confidence 기준으로 파싱 (높을수록 좋음)
                    conf = float([x for x in header.split(',') if 'overall_confidence=' in x][0].split('=')[1])
                    lig_conf = float([x for x in header.split(',') if 'ligand_confidence=' in x][0].split('=')[1])
                    seq_rec = float([x for x in header.split(',') if 'seq_rec=' in x][0].split('=')[1])
                    seq_id = int([x for x in header.split(',') if ' id=' in x][0].split('=')[1])
                    seqs.append({
                        'sample': sample_name,
                        'id': seq_id,
                        'overall_confidence': conf,
                        'ligand_confidence': lig_conf,
                        'seq_rec': seq_rec,
                        'seq': seq
                    })
                except:
                    pass

        # overall_confidence 내림차순 정렬 (높을수록 좋음)
        seqs.sort(key=lambda x: x['overall_confidence'], reverse=True)

        print(f'  생성된 서열 수: {len(seqs)}')
        print(f'  {"순위":<5} {"id":<5} {"overall_conf":<14} {"ligand_conf":<13} {"seq_rec":<10} {"길이"}')
        print('  ' + '-'*55)
        for rank, s in enumerate(seqs[:5], 1):
            print(f'  {rank:<5} {s["id"]:<5} {s["overall_confidence"]:<14.4f} {s["ligand_confidence"]:<13.4f} {s["seq_rec"]:<10.4f} {len(s["seq"])}')
        if len(seqs) > 5:
            print(f'  ... 외 {len(seqs)-5}개')

        all_results.extend(seqs)

# 전체 통합 상위 (ColabFold 후보 선정용)
if all_results:
    all_results.sort(key=lambda x: x['overall_confidence'], reverse=True)
    print(f'\n=== 전체 통합 상위 10개 (ColabFold 후보) ===')
    print(f'  {"순위":<5} {"sample":<12} {"id":<5} {"overall_conf":<14} {"ligand_conf":<13} {"seq_rec"}')
    print('  ' + '-'*60)
    for rank, s in enumerate(all_results[:10], 1):
        print(f'  {rank:<5} {s["sample"]:<12} {s["id"]:<5} {s["overall_confidence"]:<14.4f} {s["ligand_confidence"]:<13.4f} {s["seq_rec"]:.4f}')

---
## 5단계: ColabFold 입력 FASTA 생성

**[v2 신규 추가]**  

4단계에서 만든 all_results에서 상위 서열을 골라 ColabFold 입력용 .fasta 파일로 저장.  

**필터 기준:**  
- seq_rec >= 0.3: 너무 낮으면 backbone 구조를 무시한 서열 (의미 없음)  
- 상위 10개: 무료 Colab 기준 시간 고려  

**파일명 규칙:**  
rank01_sample_0_c0.XXX.fasta 형식으로 저장.  
ColabFold jobname으로 바로 사용 가능.

In [ ]:
import os
from google.colab import files

# seq_rec 필터: backbone을 어느 정도 반영한 서열만 선택
# 0.3 미만이면 backbone 구조를 거의 무시한 서열로 판단
SEQ_REC_MIN = 0.3
TOP_N = 10  # ColabFold에 넣을 서열 수 (무료 Colab: 시간 고려해 10개 이내 권장)

candidates = [r for r in all_results if r['seq_rec'] >= SEQ_REC_MIN][:TOP_N]

CF_INPUT_DIR = '/content/drive/MyDrive/RFdiffusionAA/output/colabfold_input'
os.makedirs(CF_INPUT_DIR, exist_ok=True)

print(f'전체 서열: {len(all_results)}개')
print(f'seq_rec >= {SEQ_REC_MIN} 통과: {len([r for r in all_results if r["seq_rec"] >= SEQ_REC_MIN])}개')
print(f'상위 {TOP_N}개 선별')
print()

saved = []
for rank, r in enumerate(candidates, 1):
    # ColabFold jobname으로 바로 쓸 수 있는 파일명
    # overall_confidence 앞 3자리만 사용 (파일명 간결하게)
    seq_name = f"rank{rank:02d}_{r['sample']}_c{r['overall_confidence']:.3f}"
    fasta_path = f'{CF_INPUT_DIR}/{seq_name}.fasta'

    with open(fasta_path, 'w') as f:
        f.write(f'>{seq_name}\n{r["seq"]}\n')

    saved.append(fasta_path)
    print(f'rank{rank:02d}  overall_conf={r["overall_confidence"]:.4f}  seq_rec={r["seq_rec"]:.4f}  len={len(r["seq"])}')
    print(f'       -> {seq_name}.fasta')

print(f'\n✅ {len(saved)}개 Drive 저장 완료')
print(f'경로: {CF_INPUT_DIR}')

# 로컬 다운로드 (다음 세션 ColabFold에 바로 업로드 가능)
print('\n로컬 다운로드 시작...')
for fasta_path in saved:
    files.download(fasta_path)
print('\n다음: ColabFold 노트북에 각 서열 붙여넣기')

---
## 트러블슈팅 기록

| # | 에러/문제 | 원인 | 해결 |
|---|-----------|------|------|
| 1 | ModuleNotFoundError: prody | 기본 설치에 포함 안 됨 | pip install prody |
| 2 | ModuleNotFoundError: ml_collections | openfold 의존성 누락 | pip install ml_collections |
| 3 | AttributeError: np.int | numpy 최신 버전에서 np.int 삭제됨 | sed로 residue_constants.py 직접 수정 |
| 4 | numpy 다운그레이드 실패 | Python 3.12 환경에서 구버전 numpy 빌드 불가 | 다운그레이드 대신 소스 파일 직접 수정 |
| 5 | num_ligand_res=0 | --chains_to_design 미지정으로 STR(Chain B) 리간드 미인식 | --chains_to_design "A" 추가 (v2 수정) |
| 6 | 결과 파싱 실패 (서열 0개) | 헤더가 score= 아닌 overall_confidence= 형식임 | overall_confidence 기준 파싱으로 변경 (v2 수정) |

---
## 다음 세션 (Step 4: ColabFold)

5단계에서 생성한 rank별 .fasta 파일을 ColabFold에 입력.  
ColabFold 노트북: https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb  

**필터링 기준:**  
- pTM > 0.5: 전체 구조 신뢰도  
- mean_pAE < 10: 단백질-리간드 인터페이스 오차  
- 최종 3~5개 선별 -> PyMOL 시각화  

진행 현황: Step 1 완료 | Step 2 완료 | Step 3 이번 세션 | Step 4~6 예정